# Week 4 CHALLENGES 

For this high performance coding solution
* Try adding more models
* Try making an Agentic version

3 new exciting code generation ideas
* A code tool that automatically adds docstring / comments
* A code gen tool that writes unit tests cases
* A code generator that writes trading code to buy and sell equites in a simulated environment, based on a given API


Features:

    - Ask LLM to generate a Trading Simulator in Python, paste a code, or run a sample
    - Generated codes will include self-timing and simple test
    - Port and run Python code to C++ 

In [8]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

In [10]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [11]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [12]:
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-flash-lite", "qwen2.5-coder", "llama3.2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "gemini-2.5-flash-lite": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "llama3.2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}


In [6]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

In [14]:
# You may need to overwrite this with commands from day3

compile_command = ["clang++", "-std=c++20", "-O3", "-flto=thin", "-DNDEBUG", "-o", "main", "main.cpp"] 
run_command = ["./main"]


In [25]:
sample_code = """
import time

def trading_simulator(starting_cash, trades):

    cash = starting_cash
    portfolio = {}  # Format: {'SYMBOL': quantity}
    
    print(f"--- Simulation Started (Starting Cash: ${starting_cash:,.2f}) ---")
    
    # Start high-precision timer
    start_time = time.perf_counter()
    
    # Process the trades
    for trade in trades:
        action = trade['action'].lower()
        symbol = trade['symbol'].upper()
        qty = trade['qty']
        price = trade['price']
        cost = qty * price

        if action == 'buy':
            if cash >= cost:
                cash -= cost
                portfolio[symbol] = portfolio.get(symbol, 0) + qty
                print(f"BOUGHT: {qty} shares of {symbol} at ${price:.2f}")
            else:
                print(f"REJECTED: Insufficient funds to buy {symbol}")

        elif action == 'sell':
            if portfolio.get(symbol, 0) >= qty:
                cash += cost
                portfolio[symbol] -= qty
                print(f"SOLD: {qty} shares of {symbol} at ${price:.2f}")
            else:
                print(f"REJECTED: Not enough shares of {symbol} to sell")

    # End timer
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    
    # Summary of Results
    print("--- Simulation Results ---")
    print(f"Final Cash: ${cash:,.2f}")
    print(f"Holdings: {portfolio}")
    print(f"Execution Time: {execution_time:.6f} seconds")
    
    return {"cash": cash, "portfolio": portfolio, "time": execution_time}

# ==========================================
# SAMPLE TEST 
# ==========================================
test_trades = [
    {'action': 'buy', 'symbol': 'AAPL', 'qty': 10, 'price': 175.50},
    {'action': 'buy', 'symbol': 'TSLA', 'qty': 5, 'price': 240.00},
    {'action': 'sell', 'symbol': 'AAPL', 'qty': 4, 'price': 180.00},
    {'action': 'buy', 'symbol': 'NVDA', 'qty': 2, 'price': 450.00},
    {'action': 'sell', 'symbol': 'TSLA', 'qty': 10, 'price': 250.00} # Should fail (insufficient shares)
]
result = trading_simulator(5000.00, test_trades)
"""

In [16]:
language = "C++"
extension = "cpp"

system_prompt = f"""
You are a software engineering assistant that specializes in generating programming code based on inputs.
Your task is to generate Python code, and then port or convert this Python code into a high performance C++ code. 

- For all generated or ported code, include code for tracking execution time similar to {sample_code}.  
- Respond only with Python code when generating a new Python source. 
- Respond only with C++ code when converting or porting from a Python code. 
- Do not provide any explanation as part of the code other than occasional comments.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
Use only standard headers, for example, replacing #include <bits/stdc++.h> with:
    #include <iostream>
    #include <vector>
    #include <algorithm>
    #include <string>
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with completed C++ code, and ready to test.
Python code to port:

```python
{python}
```
"""

In [17]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [18]:
def write_output(code):
    with open(f"main.cpp", "w") as f:
        f.write(code)

In [19]:
def port(model, python):
    client = clients[model]
    openai_reasoning_models = {"gpt-5"}
    reasoning_effort = "high" if model in openai_reasoning_models else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [20]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
   
    return output

In [21]:

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [ ]:

def generate_python_code(model, msg):

    create_python=f"""
        Generate a simple Python code with the following instructions: {msg}. 
        Guidelines:
        This Python code must be written so it can easily port to a C++ code.
        Include code for tracking execution time similar to {sample_code}.
        Include a separate function for quick testing without using if __name__ == "__main__":
         """     
    message = (
        [{"role": "system", "content": system_prompt}]
        + [{"role": "user", "content": create_python}]
    )

    client = clients[model]
    response = client.chat.completions.create(model=model, messages=message)
    reply = response.choices[0].message.content
    reply = reply.replace('```python','').replace('```','')
    
    return reply 

In [26]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (source)",
                value=sample_code,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"C++ (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type instructions here..  or click on example below",
            scale=4, interactive=True,
            label="Create Python",
        )
        send = gr.Button("Send", variant="primary", scale=1)
        
    with gr.Row():
        gr.Examples(
            examples=[
                ["Write a simple Python code to buy or sell equities in a simulated environment"]
            ],
            inputs=msg,
        )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

    msg.submit(fn=generate_python_code, inputs=[model, msg], outputs=[python])
    send.click(fn=generate_python_code, inputs=[model, msg], outputs=[python])

ui.launch(inbrowser=True)


# Results!
    gpt-5 Python                        0.000041 seconds

Run C++

    ollama llama3.2                     FAIL
    ollama deepseek-r1:5b               FAIL (test code ported in chinese)
    
    4th Place: gpt-5                                0.000009 seconds
    4th Place: qwen/qwen3-coder-30b-a3b-instruct    0.000008 seconds
    3rd Place: gemini-2.5-flash-lite                0.000007 seconds
    2nd Place: openai/gpt-oss-120b                  0.000006 seconds
    1st Place: claude-sonnet-4.5-20250929           0.000005 seconds

